控制无人机在二维空间飞行，躲避障碍物、应对随机风场、管理电池电量，最终到达充电站。我们比较三种强化学习智能体：

Rational（理性/风险中性） – 普通 Q-learning

Prospect（行为经济学/前景理论） – 对奖励进行非线性扭曲，比如更厌恶损失

Risk-sensitive（风险敏感） – 使用指数效用函数，重视奖励的方差

In [5]:
!pip -q install numpy matplotlib scipy tqdm

In [6]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt

from tqdm.auto import tqdm  #进度条
from scipy.stats import ttest_ind  #t检验
#t检验的目的是量化“差异是否可信”，而不是单纯看数值之差。
#p<0.05表示差异有统计学意义，可以作为结论的依据（例如：风险敏感智能体的能耗显著高于其他两者）。
#如果某个指标（如成功时间）有效样本太少，则无法进行 t 检验，需要收集更多成功 episode 或改用其他统计方法。
from matplotlib.patches import Rectangle, Circle #用于画障碍物和充电站

In [7]:
#固定随机种子
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

#通用训练参数
COMMON_CFG = { #所有智能体共享的训练参数
    "num_train_episodes": 15000,
    "num_test_episodes": 300,
    "gamma": 0.99, #折扣因子，看重未来奖励
    "alpha": 0.10,  #学习率
    "epsilon_start": 1.0, #初始探索率
    "epsilon_min": 0.01,
    "epsilon_decay": 0.997, #每回合结束后衰减
    "test_epsilon": 0.005, #测试时基本不探索  P7

    #Prospect Theory参数 P12
    "prospect_alpha": 0.88, #前景理论收益部分凸性参数
    "prospect_beta": 0.88,  #损失部分凸性参数
    "prospect_lambda": 2.35, #损失厌恶系数

    #Risk-sensitive参数
    "risk_eta": 0.5, #风险敏感系数(>0风险厌恶)

    #乐观初始化
    "optimistic_q0": 5.0, #Q表乐观初始值，鼓励探索
}

#环境参数（方案一）
ENV_CFG = {
    "name": "Scheme 1",

    "x_range": (0.0, 24.0),
    "y_range": (0.0, 12.0),

    "station_center": (21.5, 8.5),
    "station_radius": 0.9,
    "charge_power": 0.025,

    "start_mean": (3.5, 6.0),
    "start_sigma": 1.8, #起始点正态分布
    "start_clip_x": (1.0, 7.0),
    "start_clip_y": (2.0, 10.0),

    "wind_x_range": (-0.11, 0.11),
    "wind_y_range": (-0.11, 0.11),
    "wind_update_period": 15, #每15步风随机变化一次

    "obstacles": [
        (10.0, 2.0, 11.5, 5.5),
        (13.0, 7.0, 14.5, 10.0),
        (16.5, 3.5, 18.0, 6.5),
        (19.0, 9.0, 20.5, 11.5), #四个矩形障碍
    ],

    "speed_values": [0.10, 0.25, 0.40, 0.60], #可选速度
    "action_target_speeds": [0.10, 0.25, 0.40, 0.60],


    "state_bins": (14, 14, 12, 4), # #状态离散化：dx, dy, b, v

    "success_battery_threshold": 0.90, #到达时电量需>0.90
    "failure_battery_threshold": 0.04,
    "success_deadline": 140, #最多140步内成功
    "fail_step_limit": 200,

    "success_reward": 90.0,
    "discharge_penalty": -170.0,
    "step_base_reward": -1.0, #每步基础代价
    "step_speed_penalty_coef": 0.030, #速度越快额外耗能惩罚
    "step_station_bonus_coef": 0.12, #在充电站内获得额外奖励比例

    # b_{t+1} = min(1, b - 0.018v^2 - 0.003||w||^2 - 0.002|v| + I_station * 0.025)
    "battery_v2_coef": 0.018,
    "battery_w_norm2_coef": 0.003,
    "battery_abs_v_coef": 0.002,
}

print("当前方案:", ENV_CFG["name"])
print("训练 episodes:", COMMON_CFG["num_train_episodes"])
print("测试 episodes:", COMMON_CFG["num_test_episodes"])
print("state bins:", ENV_CFG["state_bins"])
print("speeds:", ENV_CFG["speed_values"])

当前方案: Scheme 1
训练 episodes: 15000
测试 episodes: 300
state bins: (14, 14, 12, 4)
speeds: [0.1, 0.25, 0.4, 0.6]


In [8]:
def clip(v, low, high):  # 边界修剪
    return max(low, min(high, v))


def point_in_rect(x, y, rect):  # 判断点是否在矩形内
    x1, y1, x2, y2 = rect
    return (x1 <= x <= x2) and (y1 <= y <= y2)


def orientation(ax, ay, bx, by, cx, cy):
    """
    返回三点 (A, B, C) 的方向：
    0 -> 共线
    1 -> 顺时针
    2 -> 逆时针
    """
    val = (by - ay) * (cx - bx) - (bx - ax) * (cy - by)
    eps = 1e-12
    if abs(val) < eps:
        return 0
    return 1 if val > 0 else 2


def on_segment(ax, ay, bx, by, cx, cy):
    """
    判断点 B 是否在线段 AC 上
    """
    eps = 1e-12
    return (
        min(ax, cx) - eps <= bx <= max(ax, cx) + eps
        and min(ay, cy) - eps <= by <= max(ay, cy) + eps
    )


def segments_intersect(x1, y1, x2, y2, x3, y3, x4, y4):
    """
    精确判断两条线段是否相交
    线段1: (x1, y1) -> (x2, y2)
    线段2: (x3, y3) -> (x4, y4)
    """
    o1 = orientation(x1, y1, x2, y2, x3, y3)
    o2 = orientation(x1, y1, x2, y2, x4, y4)
    o3 = orientation(x3, y3, x4, y4, x1, y1)
    o4 = orientation(x3, y3, x4, y4, x2, y2)

    # 一般情况
    if o1 != o2 and o3 != o4:
        return True

    # 特殊共线情况
    if o1 == 0 and on_segment(x1, y1, x3, y3, x2, y2):
        return True
    if o2 == 0 and on_segment(x1, y1, x4, y4, x2, y2):
        return True
    if o3 == 0 and on_segment(x3, y3, x1, y1, x4, y4):
        return True
    if o4 == 0 and on_segment(x3, y3, x2, y2, x4, y4):
        return True

    return False


def segment_hits_rect(x1, y1, x2, y2, rect):
    """
    精确判断运动线段是否与矩形障碍物相交
    rect = (rx1, ry1, rx2, ry2)
    """
    rx1, ry1, rx2, ry2 = rect

    # 起点或终点在矩形内，也算碰撞
    if point_in_rect(x1, y1, rect) or point_in_rect(x2, y2, rect):
        return True

    # 矩形四条边
    edges = [
        (rx1, ry1, rx2, ry1),  # 下边
        (rx2, ry1, rx2, ry2),  # 右边
        (rx2, ry2, rx1, ry2),  # 上边
        (rx1, ry2, rx1, ry1),  # 左边
    ]

    for ex1, ey1, ex2, ey2 in edges:
        if segments_intersect(x1, y1, x2, y2, ex1, ey1, ex2, ey2):
            return True

    return False


# 以上函数用于碰撞检测：无人机不能进入障碍物。
# 如果一步移动的整条线段与矩形障碍物相交，就取消移动（留在原地）。
def in_any_obstacle(x1, y1, x2, y2, obstacles):
    for rect in obstacles:
        if segment_hits_rect(x1, y1, x2, y2, rect):
            return True
    return False


def make_uniform_bin_index(value, low, high, n_bins):
    # 将连续值映射到离散区间索引 [0, n_bins-1]
    value = np.clip(value, low, high)
    edges = np.linspace(low, high, n_bins + 1)
    idx = np.digitize(value, edges[1:-1], right=False)
    return int(np.clip(idx, 0, n_bins - 1))


# 将位置偏差、电池电量等连续变量变为离散整数索引，方便查Q表。
def nearest_speed_index(v, speed_values):
    # 找到最接近当前速度的速度等级索引
    arr = np.array(speed_values)
    return int(np.argmin(np.abs(arr - v)))


def moving_average(x, window=150):
    # 计算滑动平均，用于平滑学习曲线
    x = np.asarray(x, dtype=np.float64)
    if len(x) < window:
        return x
    kernel = np.ones(window) / window
    return np.convolve(x, kernel, mode="valid")

In [9]:
# 核心环境类 DroneEnv
# 环境模拟无人机的每一步移动，返回下一个状态、奖励、是否结束等信息。
class DroneEnv:
    def __init__(self, cfg, seed=42):  # 初始化
        self.cfg = cfg
        self.rng = np.random.default_rng(seed)  # 独立随机数生成器

        # 提取配置中的边界、充电站、障碍物、速度表等
        self.xmin, self.xmax = cfg["x_range"]
        self.ymin, self.ymax = cfg["y_range"]

        self.station_x, self.station_y = cfg["station_center"]
        self.station_radius = cfg["station_radius"]
        self.charge_power = cfg["charge_power"]

        self.speed_values = cfg["speed_values"]
        self.action_target_speeds = cfg["action_target_speeds"]

        self.directions = [0.0, np.pi / 2, np.pi, 3 * np.pi / 2]  # 右、上、左、下
        self.actions = [(theta, v) for theta in self.directions for v in self.action_target_speeds]
        self.n_actions = len(self.actions)  # 4方向 × 4速度 = 16个动作

        self.n_dx, self.n_dy, self.n_b, self.n_v = cfg["state_bins"]
        self.n_states = self.n_dx * self.n_dy * self.n_b * self.n_v

        self.reset()

    def sample_start_position(self):
        mx, my = self.cfg["start_mean"]
        sigma = self.cfg["start_sigma"]

        x = self.rng.normal(mx, sigma)
        y = self.rng.normal(my, sigma)

        x = np.clip(x, self.cfg["start_clip_x"][0], self.cfg["start_clip_x"][1])
        y = np.clip(y, self.cfg["start_clip_y"][0], self.cfg["start_clip_y"][1])
        return float(x), float(y)

    def sample_wind(self):
        wx = self.rng.uniform(*self.cfg["wind_x_range"])
        wy = self.rng.uniform(*self.cfg["wind_y_range"])
        return float(wx), float(wy)

    def is_in_station(self, x, y):
        dist = np.sqrt((x - self.station_x) ** 2 + (y - self.station_y) ** 2)
        return dist < self.station_radius

    def get_state_index(self):  # 状态表示
        dx = self.station_x - self.x  # 到充电站的 x 偏差
        dy = self.station_y - self.y  # y 偏差

        dx_idx = make_uniform_bin_index(dx, -self.xmax, self.xmax, self.n_dx)
        dy_idx = make_uniform_bin_index(dy, -self.ymax, self.ymax, self.n_dy)
        b_idx = make_uniform_bin_index(self.b, 0.0, 1.0, self.n_b)
        v_idx = nearest_speed_index(self.v, self.speed_values)

        # 将四维索引展平为一维索引
        return np.ravel_multi_index(
            (dx_idx, dy_idx, b_idx, v_idx),
            (self.n_dx, self.n_dy, self.n_b, self.n_v)
        )

    def reset(self):
        self.x, self.y = self.sample_start_position()

        # 题目未明确初始电量，这里采用满电
        self.b = 1.0
        self.v = self.speed_values[0]
        self.t = 0

        self.total_reward = 0.0
        self.total_energy_used = 0.0
        self.collision_count = 0

        self.wind = self.sample_wind()
        return self.get_state_index()

    def _refresh_wind_if_needed(self):
        period = self.cfg["wind_update_period"]
        if self.t > 0 and (self.t % period == 0):
            self.wind = self.sample_wind()

    def _apply_dynamics(self, theta, target_v):
        wx, wy = self.wind
        self.v = target_v

        nx = self.x + self.v * np.cos(theta) + wx
        ny = self.y + self.v * np.sin(theta) + wy

        # 保持在工作区域内
        nx = clip(nx, self.xmin, self.xmax)
        ny = clip(ny, self.ymin, self.ymax)

        # 碰撞规则：禁止进入障碍物
        # 如果一步运动的线段与任意矩形障碍物相交，则保持原地，并记录碰撞
        hit = in_any_obstacle(self.x, self.y, nx, ny, self.cfg["obstacles"])

        if hit:
            nx, ny = self.x, self.y
            self.collision_count += 1

        self.x, self.y = nx, ny
        return hit

    def _battery_update(self, in_station):
        wx, wy = self.wind

        energy_use = (
            self.cfg["battery_v2_coef"] * (self.v ** 2)
            + self.cfg["battery_w_norm2_coef"] * (wx ** 2 + wy ** 2)
            + self.cfg["battery_abs_v_coef"] * abs(self.v)
        )

        charge_gain = self.charge_power if in_station else 0.0

        self.b = min(1.0, self.b - energy_use + charge_gain)
        self.total_energy_used += energy_use

        return energy_use, charge_gain

    def _normal_step_reward(self, in_station):
        return (
            self.cfg["step_base_reward"]
            - self.cfg["step_speed_penalty_coef"] * (self.v ** 2)
            + self.cfg["step_station_bonus_coef"] * self.charge_power * (1 if in_station else 0)
        )

    def _is_success(self, in_station):
        return (
            in_station
            and (self.b > self.cfg["success_battery_threshold"])
            and (self.t <= self.cfg["success_deadline"])
        )

    def _is_time_fail(self):
        return self.t >= self.cfg["fail_step_limit"]

    def step(self, action_idx):  # 动力学一步
        self._refresh_wind_if_needed()  # 定期刷新随机风

        theta, target_v = self.actions[action_idx]
        collision = self._apply_dynamics(theta, target_v)  # 移动并检查碰撞

        in_station = self.is_in_station(self.x, self.y)
        energy_use, charge_gain = self._battery_update(in_station)

        self.t += 1

        # 判断是否成功 / 失败
        success = self._is_success(in_station)
        battery_fail = (self.b < self.cfg["failure_battery_threshold"])
        time_fail = self._is_time_fail()

        if success:
            reward = self.cfg["success_reward"]
            done = True
        elif battery_fail:
            reward = self.cfg["discharge_penalty"]
            done = True
        else:
            reward = self._normal_step_reward(in_station)
            done = time_fail

        self.total_reward += reward

        info = {
            "x": self.x,
            "y": self.y,
            "battery": self.b,
            "speed": self.v,
            "wind": self.wind,
            "collision": collision,
            "collision_count": self.collision_count,
            "success": success,
            "battery_fail": battery_fail,
            "time_fail": time_fail,
            "energy_used_step": energy_use,
            "charge_gain_step": charge_gain,
            "total_energy_used": self.total_energy_used,
            "t": self.t,
        }

        return self.get_state_index(), reward, done, info

电池更新：
b = b - 0.018*v² - 0.003*(wx²+wy²) - 0.002*|v| + （充电站内）0.025

风每15步重新采样一次，期间保持不变。

In [10]:
env = DroneEnv(ENV_CFG, seed=SEED)

state = env.reset()
print("初始状态索引:", state)
print("状态总数:", env.n_states)
print("动作总数:", env.n_actions)

for i in range(5):
    a = np.random.randint(env.n_actions)
    next_state, reward, done, info = env.step(a)
    print(f"step={i+1}, action={a}, reward={reward:.3f}, battery={info['battery']:.3f}, done={done}")
    if done:
        break

初始状态索引: 8540
状态总数: 9408
动作总数: 16
step=1, action=6, reward=-1.005, battery=0.996, done=False
step=2, action=3, reward=-1.011, battery=0.989, done=False
step=3, action=12, reward=-1.000, battery=0.988, done=False
step=4, action=14, reward=-1.005, battery=0.984, done=False
step=5, action=10, reward=-1.005, battery=0.981, done=False


In [11]:
#辅助选择策略（打破平局）
def one_step_lookahead_score(env, action_idx):
  #当多个动作有相同Q值时，额外评估该动作一步后的效果：
  #距离充电站更近、不碰撞、速度更低 → 分数越小越好
    theta, target_v = env.actions[action_idx]
    wx, wy = env.wind

    nx = env.x + target_v * np.cos(theta) + wx
    ny = env.y + target_v * np.sin(theta) + wy

    nx = clip(nx, env.xmin, env.xmax)
    ny = clip(ny, env.ymin, env.ymax)

    hit = in_any_obstacle(env.x, env.y, nx, ny, env.cfg["obstacles"])
    if hit:
        nx, ny = env.x, env.y

    dist_to_station = np.hypot(env.station_x - nx, env.station_y - ny)

    # 分数越小越好：更接近充电站、更少碰撞、略偏好低速
    score = dist_to_station + (4.0 if hit else 0.0) + 0.05 * target_v
    return score


class BaseTabularAgent:#智能体基类
    def __init__(self, n_states, n_actions, alpha, gamma, name="Base"):
        self.n_states = n_states
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.name = name
        self.Q = np.zeros((n_states, n_actions), dtype=np.float64) #Q表格

    def select_action(self, state, epsilon, rng, env=None):
        if rng.random() < epsilon:
            return int(rng.integers(self.n_actions))

        q = self.Q[state]
        max_q = np.max(q)
        best_actions = np.flatnonzero(np.isclose(q, max_q)) #所有Q值最大的动作

        # if 只有一个最佳动作: return 它
        if len(best_actions) == 1 or env is None:
            return int(rng.choice(best_actions))

        scored = []
        for a in best_actions:
            score = one_step_lookahead_score(env, a)
            scored.append((score, a))

        min_score = min(s for s, _ in scored)
        best_scored_actions = [a for s, a in scored if np.isclose(s, min_score)]
        return int(rng.choice(best_scored_actions))

   #Rational(普通Q-learning)
    def update(self, state, action, reward, next_state, done):
        raise NotImplementedError


class RationalQLearningAgent(BaseTabularAgent):
    def __init__(self, n_states, n_actions, alpha, gamma):
        super().__init__(n_states, n_actions, alpha, gamma, name="Rational Q-learning")

    def update(self, state, action, reward, next_state, done):
        if done:
            target = reward
        else:
            target = reward + self.gamma * np.max(self.Q[next_state])

        td_error = target - self.Q[state, action]
        self.Q[state, action] += self.alpha * td_error


class ProspectQLearningAgent(BaseTabularAgent):
    def __init__(self, n_states, n_actions, alpha, gamma, alpha_p, beta_p, lambda_p):
        super().__init__(n_states, n_actions, alpha, gamma, name="Prospect Theory Q-learning")
        self.alpha_p = alpha_p
        self.beta_p = beta_p
        self.lambda_p = lambda_p
     #Prospect Theory Q-learning
    def value_function(self, r):
        if r >= 0:
            return r ** self.alpha_p
        else:
            return -self.lambda_p * ((-r) ** self.beta_p)
            #if r>=0: return r**0.88
            #else: return -2.35 * ((-r)**0.88)   #损失被放大且非线性

    def update(self, state, action, reward, next_state, done):
        subjective_reward = self.value_function(reward) #扭曲原始奖励

        if done:
            target = subjective_reward
        else:
            target = subjective_reward + self.gamma * np.max(self.Q[next_state])

        td_error = target - self.Q[state, action]
        self.Q[state, action] += self.alpha * td_error


class RiskSensitiveQLearningAgent(BaseTabularAgent): #Risk-sensitive Q-learning（指数效用）
    def __init__(self, n_states, n_actions, alpha, gamma, eta):
        super().__init__(n_states, n_actions, alpha, gamma, name="Risk-sensitive Q-learning")
        self.eta = eta
        self.U = np.ones((n_states, n_actions), dtype=np.float64) #存储exp(-η Q(s,a))

    def update(self, state, action, reward, next_state, done):
        old_u = self.U[state, action]
#目标U = exp(-η * r) * (min_{a'} U(next, a'))^γ
        if done:
            log_target_u = -self.eta * reward
        else:
            min_next_u = np.min(self.U[next_state])
            min_next_u = max(min_next_u, 1e-300)
            log_target_u = -self.eta * reward + self.gamma * np.log(min_next_u)

        log_target_u = np.clip(log_target_u, -700, 700)
        target_u = np.exp(log_target_u)

        new_u = (1.0 - self.alpha) * old_u + self.alpha * target_u
        new_u = np.clip(new_u, 1e-300, 1e300)

        self.U[state, action] = new_u
        self.Q[state, action] = -(1.0 / self.eta) * np.log(new_u) #反向算出Q值

In [12]:
def build_agent(agent_type, env, cfg):
    if agent_type == "rational":
        return RationalQLearningAgent(
            n_states=env.n_states,
            n_actions=env.n_actions,
            alpha=cfg["alpha"],
            gamma=cfg["gamma"],
        )
    elif agent_type == "prospect":
        return ProspectQLearningAgent(
            n_states=env.n_states,
            n_actions=env.n_actions,
            alpha=cfg["alpha"],
            gamma=cfg["gamma"],
            alpha_p=cfg["prospect_alpha"],
            beta_p=cfg["prospect_beta"],
            lambda_p=cfg["prospect_lambda"],
        )
    elif agent_type == "risk":
        return RiskSensitiveQLearningAgent(
            n_states=env.n_states,
            n_actions=env.n_actions,
            alpha=cfg["alpha"],
            gamma=cfg["gamma"],
            eta=cfg["risk_eta"],
        )
    else:
        raise ValueError(f"未知 agent_type: {agent_type}")


def optimistic_init(agent, q0=5.0):
    agent.Q.fill(q0)
    if hasattr(agent, "U"):
        agent.U[:] = np.exp(np.clip(-agent.eta * q0, -700, 700))

In [13]:
def train_agent(agent, env_cfg, cfg, seed=42, log_every=500):
    env = DroneEnv(env_cfg, seed=seed)
    rng = np.random.default_rng(seed + 999)

    epsilon = cfg["epsilon_start"]
    num_episodes = cfg["num_train_episodes"]

    history = {
        "episode_reward": [],
        "episode_length": [],
        "episode_collisions": [],
        "final_battery": [],
        "success": [],
        "epsilon": [],
    }

    for ep in tqdm(range(1, num_episodes + 1), desc=f"Training {agent.name}"):
        state = env.reset()
        done = False
        last_info = None

        while not done:
            action = agent.select_action(state, epsilon, rng, env)
            next_state, reward, done, info = env.step(action)

            agent.update(state, action, reward, next_state, done)

            state = next_state
            last_info = info

        history["episode_reward"].append(env.total_reward)
        history["episode_length"].append(last_info["t"])
        history["episode_collisions"].append(last_info["collision_count"])
        history["final_battery"].append(last_info["battery"])
        history["success"].append(1 if last_info["success"] else 0)
        history["epsilon"].append(epsilon)

        epsilon = max(cfg["epsilon_min"], epsilon * cfg["epsilon_decay"])

        if ep % log_every == 0:
            recent_success = np.mean(history["success"][-200:]) * 100
            recent_reward = np.mean(history["episode_reward"][-200:])
            print(
                f"Episode {ep:5d} | epsilon={epsilon:.4f} | "
                f"recent_reward={recent_reward:.3f} | recent_success={recent_success:.1f}%"
            )

    return history

In [14]:
def evaluate_agent(agent, env_cfg, cfg, seed_base=2024, record_trajectories=4):
    num_episodes = cfg["num_test_episodes"]
    epsilon = cfg["test_epsilon"]

    results = {
        "success": [],
        "episode_length": [],
        "success_steps": [],
        "energy_used": [],
        "collision_episode": [],
        "collision_count": [],
        "final_battery": [],
        "final_x": [],
        "final_y": [],
        "trajectories": [],
    }

    for ep in tqdm(range(num_episodes), desc=f"Testing {agent.name}"):
        env = DroneEnv(env_cfg, seed=seed_base + ep)
        rng = np.random.default_rng(seed_base + 10000 + ep)

        state = env.reset()
        done = False

        path = [(env.x, env.y, env.b)]
        last_info = None

        while not done:
            action = agent.select_action(state, epsilon, rng, env)
            next_state, reward, done, info = env.step(action)
            state = next_state
            last_info = info
            path.append((info["x"], info["y"], info["battery"]))

        success = 1 if last_info["success"] else 0

        results["success"].append(success)
        results["episode_length"].append(last_info["t"])
        results["energy_used"].append(last_info["total_energy_used"])
        results["collision_episode"].append(1 if last_info["collision_count"] > 0 else 0)
        results["collision_count"].append(last_info["collision_count"])
        results["final_battery"].append(last_info["battery"])
        results["final_x"].append(last_info["x"])
        results["final_y"].append(last_info["y"])

        if success:
            results["success_steps"].append(last_info["t"])

        if len(results["trajectories"]) < record_trajectories:
            results["trajectories"].append({
                "episode_id": ep + 1,
                "path": path,
                "success": bool(success),
                "steps": last_info["t"],
            })

    summary = {
        "success_rate_percent": 100 * np.mean(results["success"]),
        "mean_episode_length": float(np.mean(results["episode_length"])),
        "mean_success_time": float(np.mean(results["success_steps"])) if len(results["success_steps"]) > 0 else np.nan,
        "mean_energy_used": float(np.mean(results["energy_used"])),
        "collision_episode_percent": 100 * np.mean(results["collision_episode"]),
        "mean_final_battery": float(np.mean(results["final_battery"])),
    }

    return results, summary

In [15]:
def print_summary_table(summaries):
    print("=" * 95)
    print(f"{'Agent':<28} {'Success%':>10} {'MeanSuccTime':>15} {'MeanEnergy':>15} {'Collision%':>12} {'MeanBattery':>12}")
    print("=" * 95)

    for name, s in summaries.items():
        print(
            f"{name:<28} "
            f"{s['success_rate_percent']:>10.2f} "
            f"{s['mean_success_time']:>15.2f} "
            f"{s['mean_energy_used']:>15.4f} "
            f"{s['collision_episode_percent']:>12.2f} "
            f"{s['mean_final_battery']:>12.4f}"
        )
    print("=" * 95)


def draw_env_layout(ax, env_cfg):
    ax.set_xlim(*env_cfg["x_range"])
    ax.set_ylim(*env_cfg["y_range"])
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.grid(True, alpha=0.3)

    for rect in env_cfg["obstacles"]:
        x1, y1, x2, y2 = rect
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, hatch='///'))

    cx, cy = env_cfg["station_center"]
    r = env_cfg["station_radius"]
    ax.add_patch(Circle((cx, cy), r, fill=False))
    ax.scatter([cx], [cy], marker="+", s=120)


def plot_learning_curves(histories, window=150):
    plt.figure(figsize=(10, 5))
    for name, history in histories.items():
        y = moving_average(history["episode_reward"], window=window)
        x = np.arange(len(y)) + window
        plt.plot(x, y, label=name)
    plt.xlabel("Episode")
    plt.ylabel(f"Moving Avg Reward (window={window})")
    plt.title("Learning Curves")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_success_time_cdf(results_dict):
    plt.figure(figsize=(10, 5))
    for name, result in results_dict.items():
        times = np.array(result["success_steps"], dtype=np.float64)
        if len(times) == 0:
            continue
        times = np.sort(times)
        cdf = np.arange(1, len(times) + 1) / len(times)
        plt.plot(times, cdf, label=name)
    plt.xlabel("Steps to Success")
    plt.ylabel("CDF")
    plt.title("CDF of Success Time")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_final_battery_heatmap(result, env_cfg, title, nx=20, ny=12):
    xs = np.array(result["final_x"])
    ys = np.array(result["final_y"])
    bs = np.array(result["final_battery"])

    xmin, xmax = env_cfg["x_range"]
    ymin, ymax = env_cfg["y_range"]

    xedges = np.linspace(xmin, xmax, nx + 1)
    yedges = np.linspace(ymin, ymax, ny + 1)

    sum_grid = np.zeros((ny, nx), dtype=np.float64)
    cnt_grid = np.zeros((ny, nx), dtype=np.float64)

    x_idx = np.clip(np.digitize(xs, xedges) - 1, 0, nx - 1)
    y_idx = np.clip(np.digitize(ys, yedges) - 1, 0, ny - 1)

    for xi, yi, b in zip(x_idx, y_idx, bs):
        sum_grid[yi, xi] += b
        cnt_grid[yi, xi] += 1

    mean_grid = np.divide(
        sum_grid,
        cnt_grid,
        out=np.full_like(sum_grid, np.nan),
        where=cnt_grid > 0
    )

    fig, ax = plt.subplots(figsize=(10, 4))
    im = ax.imshow(
        mean_grid,
        origin="lower",
        extent=[xmin, xmax, ymin, ymax],
        aspect="auto"
    )
    draw_env_layout(ax, env_cfg)
    ax.set_title(f"Final Battery Heatmap - {title}")
    plt.colorbar(im, ax=ax, label="Mean Final Battery")
    plt.show()


def plot_trajectories(result, env_cfg, title):
    plt.figure(figsize=(10, 4))
    ax = plt.gca()
    draw_env_layout(ax, env_cfg)

    for traj in result["trajectories"]:
        path = np.array([(x, y) for x, y, b in traj["path"]], dtype=np.float64)
        ax.plot(path[:, 0], path[:, 1], label=f"ep{traj['episode_id']} | succ={traj['success']}")
        ax.scatter(path[0, 0], path[0, 1], s=30)
        ax.scatter(path[-1, 0], path[-1, 1], s=50, marker="x")

    ax.set_title(f"Trajectories - {title}")
    ax.legend()
    plt.show()


def pairwise_ttest(metric_name, results_dict):
    names = list(results_dict.keys())
    print(f"\n=== Welch t-test for: {metric_name} ===")
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            name1 = names[i]
            name2 = names[j]

            x = np.array(results_dict[name1][metric_name], dtype=np.float64)
            y = np.array(results_dict[name2][metric_name], dtype=np.float64)

            stat, p = ttest_ind(x, y, equal_var=False, nan_policy="omit")
            sig = "YES" if p < 0.05 else "NO"

            print(
                f"{name1:>15} vs {name2:<15} | "
                f"t={stat:>9.4f} | p={p:>12.6g} | significant={sig}"
            )

In [16]:
# =========================
# Repeated runs: config + helpers
# 放在 pairwise_ttest() 后面，smoke-test 前面
# =========================

from copy import deepcopy
import numpy as np

# 每个 agent 独立重复训练/测试的次数
N_RUNS = 5

# 每次 run 的随机种子
RUN_SEEDS = [100, 200, 300, 400, 500]

# 正式实验仍然使用原来的 COMMON_CFG
REPEATED_CFG = COMMON_CFG.copy()

print("Repeated experiment config:")
print("N_RUNS =", N_RUNS)
print("RUN_SEEDS =", RUN_SEEDS)
print("Train episodes per run =", REPEATED_CFG["num_train_episodes"])
print("Test episodes per run  =", REPEATED_CFG["num_test_episodes"])


def aggregate_summaries(summary_list):
    """
    将多个 run 的 summary 聚合为 mean/std
    """
    metric_names = [
        "success_rate_percent",
        "mean_episode_length",
        "mean_success_time",
        "mean_energy_used",
        "collision_episode_percent",
        "mean_final_battery",
    ]

    agg = {}
    for m in metric_names:
        vals = np.array([s[m] for s in summary_list], dtype=np.float64)
        agg[m] = {
            "mean": float(np.nanmean(vals)),
            "std": float(np.nanstd(vals, ddof=0)),
            "raw": vals,
        }
    return agg


def print_aggregated_summary_table(agg_dict):
    print("=" * 135)
    print(
        f"{'Agent':<28} "
        f"{'Success % (mean±std)':>22} "
        f"{'MeanSuccTime (mean±std)':>24} "
        f"{'MeanEnergy (mean±std)':>24} "
        f"{'Collision % (mean±std)':>24} "
        f"{'MeanBattery (mean±std)':>24}"
    )
    print("=" * 135)

    for name, agg in agg_dict.items():
        succ = agg["success_rate_percent"]
        succ_time = agg["mean_success_time"]
        energy = agg["mean_energy_used"]
        coll = agg["collision_episode_percent"]
        batt = agg["mean_final_battery"]

        def fmt2(ms):
            if np.isnan(ms["mean"]):
                return "nan ± nan"
            return f"{ms['mean']:.2f} ± {ms['std']:.2f}"

        def fmt4(ms):
            if np.isnan(ms["mean"]):
                return "nan ± nan"
            return f"{ms['mean']:.4f} ± {ms['std']:.4f}"

        print(
            f"{name:<28} "
            f"{fmt2(succ):>22} "
            f"{fmt2(succ_time):>24} "
            f"{fmt4(energy):>24} "
            f"{fmt2(coll):>24} "
            f"{fmt4(batt):>24}"
        )

    print("=" * 135)


def merge_results_across_runs(results_list):
    """
    把多个 run 的 results 合并成一个大 results dict，
    便于做 t-test 或统一分析。
    """
    merged = {}
    if len(results_list) == 0:
        return merged

    keys = results_list[0].keys()
    for k in keys:
        merged[k] = []

    for r in results_list:
        for k in keys:
            if isinstance(r[k], list):
                merged[k].extend(r[k])
            else:
                merged[k].append(r[k])

    return merged


def run_repeated_experiments(env_cfg, cfg, n_runs=5, run_seeds=None):
    """
    对三个 agent 分别做 n_runs 次独立训练+测试
    """
    env_template = DroneEnv(env_cfg, seed=SEED)

    all_data = {
        "Rational Q-learning": {
            "agent_type": "rational",
            "histories": [],
            "results": [],
            "summaries": [],
        },
        "Prospect Q-learning": {
            "agent_type": "prospect",
            "histories": [],
            "results": [],
            "summaries": [],
        },
        "Risk-sensitive Q-learning": {
            "agent_type": "risk",
            "histories": [],
            "results": [],
            "summaries": [],
        },
    }

    if run_seeds is None:
        run_seeds = [1000 + i * 100 for i in range(n_runs)]

    for run_id in range(n_runs):
        base_seed = run_seeds[run_id]
        print(f"\n================ RUN {run_id + 1}/{n_runs} | base_seed={base_seed} ================\n")

        for agent_name, pack in all_data.items():
            agent_type = pack["agent_type"]

            agent = build_agent(agent_type, env_template, cfg)
            optimistic_init(agent, q0=cfg["optimistic_q0"])

            history = train_agent(
                agent=agent,
                env_cfg=env_cfg,
                cfg=cfg,
                seed=base_seed + 1,
                log_every=1000
            )

            results, summary = evaluate_agent(
                agent=agent,
                env_cfg=env_cfg,
                cfg=cfg,
                seed_base=base_seed + 3000
            )

            pack["histories"].append(history)
            pack["results"].append(results)
            pack["summaries"].append(summary)

            print(agent_name, "summary:", summary)

    return all_data

Repeated experiment config:
N_RUNS = 5
RUN_SEEDS = [100, 200, 300, 400, 500]
Train episodes per run = 15000
Test episodes per run  = 300


In [17]:
SMOKE_CFG = COMMON_CFG.copy()
SMOKE_CFG["num_train_episodes"] = 2000 #用2000个episode确认代码无错误
SMOKE_CFG["num_test_episodes"] = 100

env_tmp = DroneEnv(ENV_CFG, seed=SEED)

agent_r_smoke = build_agent("rational", env_tmp, SMOKE_CFG)
agent_p_smoke = build_agent("prospect", env_tmp, SMOKE_CFG)
agent_s_smoke = build_agent("risk", env_tmp, SMOKE_CFG)

optimistic_init(agent_r_smoke, q0=SMOKE_CFG["optimistic_q0"])
optimistic_init(agent_p_smoke, q0=SMOKE_CFG["optimistic_q0"])
optimistic_init(agent_s_smoke, q0=SMOKE_CFG["optimistic_q0"])

history_r_smoke = train_agent(agent_r_smoke, ENV_CFG, SMOKE_CFG, seed=SEED + 1, log_every=500)
history_p_smoke = train_agent(agent_p_smoke, ENV_CFG, SMOKE_CFG, seed=SEED + 2, log_every=500)
history_s_smoke = train_agent(agent_s_smoke, ENV_CFG, SMOKE_CFG, seed=SEED + 3, log_every=500)

results_r_smoke, summary_r_smoke = evaluate_agent(agent_r_smoke, ENV_CFG, SMOKE_CFG, seed_base=4000)
results_p_smoke, summary_p_smoke = evaluate_agent(agent_p_smoke, ENV_CFG, SMOKE_CFG, seed_base=4000)
results_s_smoke, summary_s_smoke = evaluate_agent(agent_s_smoke, ENV_CFG, SMOKE_CFG, seed_base=4000)

print("Rational:", summary_r_smoke)
print("Prospect:", summary_p_smoke)
print("Risk    :", summary_s_smoke)

Training Rational Q-learning:   0%|          | 0/2000 [00:00<?, ?it/s]

Episode   500 | epsilon=0.2226 | recent_reward=-200.085 | recent_success=0.5%
Episode  1000 | epsilon=0.0496 | recent_reward=-199.928 | recent_success=0.5%
Episode  1500 | epsilon=0.0110 | recent_reward=-200.901 | recent_success=0.0%
Episode  2000 | epsilon=0.0100 | recent_reward=-200.906 | recent_success=0.0%


Training Prospect Theory Q-learning:   0%|          | 0/2000 [00:00<?, ?it/s]

Episode   500 | epsilon=0.2226 | recent_reward=-200.895 | recent_success=0.0%
Episode  1000 | epsilon=0.0496 | recent_reward=-199.021 | recent_success=1.0%
Episode  1500 | epsilon=0.0110 | recent_reward=-200.896 | recent_success=0.0%
Episode  2000 | epsilon=0.0100 | recent_reward=-200.904 | recent_success=0.0%


Training Risk-sensitive Q-learning:   0%|          | 0/2000 [00:00<?, ?it/s]

Episode   500 | epsilon=0.2226 | recent_reward=-200.021 | recent_success=0.5%
Episode  1000 | epsilon=0.0496 | recent_reward=-200.893 | recent_success=0.0%
Episode  1500 | epsilon=0.0110 | recent_reward=-200.898 | recent_success=0.0%
Episode  2000 | epsilon=0.0100 | recent_reward=-200.898 | recent_success=0.0%


Testing Rational Q-learning:   0%|          | 0/100 [00:00<?, ?it/s]

Testing Prospect Theory Q-learning:   0%|          | 0/100 [00:00<?, ?it/s]

Testing Risk-sensitive Q-learning:   0%|          | 0/100 [00:00<?, ?it/s]

Rational: {'success_rate_percent': np.float64(0.0), 'mean_episode_length': 200.0, 'mean_success_time': nan, 'mean_energy_used': 0.5651304528381922, 'collision_episode_percent': np.float64(17.0), 'mean_final_battery': 0.43486954716180753}
Prospect: {'success_rate_percent': np.float64(1.0), 'mean_episode_length': 198.48, 'mean_success_time': 48.0, 'mean_energy_used': 0.47957383996866126, 'collision_episode_percent': np.float64(27.0), 'mean_final_battery': 0.5348206161443477}
Risk    : {'success_rate_percent': np.float64(0.0), 'mean_episode_length': 200.0, 'mean_success_time': nan, 'mean_energy_used': 0.5407702528381921, 'collision_episode_percent': np.float64(43.0), 'mean_final_battery': 0.5258104931714516}


In [ ]:
# =========================
# 正式 repeated runs 主实验
# 替换原来的“每个智能体跑15000episodes”那个 cell
# =========================

all_runs_data = run_repeated_experiments(
    env_cfg=ENV_CFG,
    cfg=REPEATED_CFG,
    n_runs=N_RUNS,
    run_seeds=RUN_SEEDS
)

print("所有 repeated runs 已完成。")


================ RUN 1/5 | base_seed=100 ================



Training Rational Q-learning:   0%|          | 0/15000 [00:00<?, ?it/s]

Episode  1000 | epsilon=0.0496 | recent_reward=-200.897 | recent_success=0.0%
Episode  2000 | epsilon=0.0100 | recent_reward=-200.903 | recent_success=0.0%
Episode  3000 | epsilon=0.0100 | recent_reward=-200.912 | recent_success=0.0%
Episode  4000 | epsilon=0.0100 | recent_reward=-200.920 | recent_success=0.0%
Episode  5000 | epsilon=0.0100 | recent_reward=-200.909 | recent_success=0.0%
Episode  6000 | epsilon=0.0100 | recent_reward=-200.925 | recent_success=0.0%
Episode  7000 | epsilon=0.0100 | recent_reward=-200.919 | recent_success=0.0%
Episode  8000 | epsilon=0.0100 | recent_reward=-200.917 | recent_success=0.0%
Episode  9000 | epsilon=0.0100 | recent_reward=-200.926 | recent_success=0.0%
Episode 10000 | epsilon=0.0100 | recent_reward=-200.922 | recent_success=0.0%
Episode 11000 | epsilon=0.0100 | recent_reward=-200.928 | recent_success=0.0%
Episode 12000 | epsilon=0.0100 | recent_reward=-200.930 | recent_success=0.0%
Episode 13000 | epsilon=0.0100 | recent_reward=-200.180 | recent

Testing Rational Q-learning:   0%|          | 0/300 [00:00<?, ?it/s]

Rational Q-learning summary: {'success_rate_percent': np.float64(0.0), 'mean_episode_length': 200.0, 'mean_success_time': nan, 'mean_energy_used': 0.5243053749946519, 'collision_episode_percent': np.float64(59.333333333333336), 'mean_final_battery': 0.48081458354388296}


...

Episode  1000 | epsilon=0.0496 | recent_reward=-199.853 | recent_success=0.5%
Episode  2000 | epsilon=0.0100 | recent_reward=-200.905 | recent_success=0.0%
Episode  3000 | epsilon=0.0100 | recent_reward=-199.712 | recent_success=0.5%
Episode  4000 | epsilon=0.0100 | recent_reward=-200.915 | recent_success=0.0%
Episode  5000 | epsilon=0.0100 | recent_reward=-200.920 | recent_success=0.0%
Episode  6000 | epsilon=0.0100 | recent_reward=-200.924 | recent_success=0.0%
Episode  7000 | epsilon=0.0100 | recent_reward=-200.928 | recent_success=0.0%
Episode  8000 | epsilon=0.0100 | recent_reward=-200.926 | recent_success=0.0%
Episode  9000 | epsilon=0.0100 | recent_reward=-200.916 | recent_success=0.0%
Episode 10000 | epsilon=0.0100 | recent_reward=-200.931 | recent_success=0.0%
Episode 11000 | epsilon=0.0100 | recent_reward=-200.148 | recent_success=0.5%
Episode 12000 | epsilon=0.0100 | recent_reward=-200.098 | recent_success=0.5%
Episode 13000 | epsilon=0.0100 | recent_reward=-200.943 | recent

Testing Prospect Theory Q-learning:   0%|          | 0/300 [00:00<?, ?it/s]

Prospect Q-learning summary: {'success_rate_percent': np.float64(0.0), 'mean_episode_length': 199.98, 'mean_success_time': nan, 'mean_energy_used': 0.6160654158152121, 'collision_episode_percent': np.float64(35.0), 'mean_final_battery': 0.3912679175181211}


Training Risk-sensitive Q-learning:   0%|          | 0/15000 [00:00<?, ?it/s]

Episode  1000 | epsilon=0.0496 | recent_reward=-200.897 | recent_success=0.0%
Episode  2000 | epsilon=0.0100 | recent_reward=-200.893 | recent_success=0.0%
Episode  3000 | epsilon=0.0100 | recent_reward=-200.895 | recent_success=0.0%
Episode  4000 | epsilon=0.0100 | recent_reward=-200.904 | recent_success=0.0%
Episode  5000 | epsilon=0.0100 | recent_reward=-200.903 | recent_success=0.0%
Episode  6000 | epsilon=0.0100 | recent_reward=-200.901 | recent_success=0.0%
Episode  7000 | epsilon=0.0100 | recent_reward=-200.907 | recent_success=0.0%
Episode  8000 | epsilon=0.0100 | recent_reward=-200.906 | recent_success=0.0%
Episode  9000 | epsilon=0.0100 | recent_reward=-200.906 | recent_success=0.0%
Episode 10000 | epsilon=0.0100 | recent_reward=-200.905 | recent_success=0.0%
Episode 11000 | epsilon=0.0100 | recent_reward=-200.908 | recent_success=0.0%


In [ ]:
# =========================
# 聚合多次 runs 的 summary
# 替换原来的“每个智能体测试300episodes”那个 cell
# =========================

aggregated = {}
for agent_name, pack in all_runs_data.items():
    aggregated[agent_name] = aggregate_summaries(pack["summaries"])

print("多次独立运行的聚合结果已准备完成。")
for agent_name, agg in aggregated.items():
    print(agent_name)
    for k, v in agg.items():
        print(f"  {k}: mean={v['mean']:.6f}, std={v['std']:.6f}")

In [ ]:
# =========================
# 打印 mean ± std 总表
# 替换原来的 summaries = {...}; print_summary_table(summaries)
# =========================

print_aggregated_summary_table(aggregated)

理性智能体成功率0%：完全学不会到达充电站。这可能是由于：

环境随机风+障碍物太困难，稀疏的正奖励（90）被大量负步骤奖励淹没。

探索不足（ε衰减过快）或学习率不合适。

前景理论智能体成功率8%：虽然很低，但确实是唯一起到一定效果的。前景理论的损失厌恶函数(λ=2.35)让智能体更害怕失败（如碰撞或低电量），从而可能更谨慎地移动，偶然找到安全路径。

风险敏感智能体成功率0%：η=0.5的风险厌恶没有帮助，反而使能量消耗最高（0.567），最终电量最低。说明指数效用准则在此环境下可能过于保守，导致智能体不敢冒险接近充电站。

备注：nan 出现在 Rational 和 Risk 的 MeanSuccTime，因为它们从未成功，无法计算平均成功时间。

In [ ]:
# =========================
# 合并多个 run 的 results，准备 t-test
# 替换原来的 results_dict = {...}
# =========================

merged_results_dict = {}
for agent_name, pack in all_runs_data.items():
    merged_results_dict[agent_name] = merge_results_across_runs(pack["results"])

print("Merged results prepared for t-test.")
for k, v in merged_results_dict.items():
    print(k, "merged test episodes =", len(v["success"]))

In [ ]:
# =========================
# t-test by energy_used
# 替换原来的 pairwise_ttest("energy_used", results_dict)
# =========================

pairwise_ttest("energy_used", merged_results_dict)

In [ ]:
# =========================
# 选一个 representative run 来画图
# 放在画图 cell 前面
# =========================

rep_run_idx = 0  # 用第 1 次 run 作为代表性可视化

history_rational = all_runs_data["Rational Q-learning"]["histories"][rep_run_idx]
history_prospect = all_runs_data["Prospect Q-learning"]["histories"][rep_run_idx]
history_risk = all_runs_data["Risk-sensitive Q-learning"]["histories"][rep_run_idx]

results_rational = all_runs_data["Rational Q-learning"]["results"][rep_run_idx]
results_prospect = all_runs_data["Prospect Q-learning"]["results"][rep_run_idx]
results_risk = all_runs_data["Risk-sensitive Q-learning"]["results"][rep_run_idx]

print(f"Representative run selected: run #{rep_run_idx + 1}")

In [ ]:
histories = {
    "Rational": history_rational,
    "Prospect": history_prospect,
    "Risk-sensitive": history_risk,
}
plot_learning_curves(histories, window=150)
#X轴：episode编号，Y轴：最近150个episode的平均总奖励
#可以帮助观察智能体是否在学习、奖励是否收敛

学习曲线（滑动平均奖励）
三条曲线几乎全部稳定在 -200 左右。为什么？

每个失败episode（电池耗尽或超时）获得 -170 的惩罚，再加上每步约 -1 的代价，一个长episode（200步）总奖励 ≈ -200。成功episode 能得到 +90 但会提前结束，所以平均奖励接近 -200 说明绝大多数episode失败了。

Prospect 偶尔有成功，被平均后和 -200 差异很小，所以曲线看不出上升。

结论：环境对三个智能体都非常困难，几乎学不会真正成功到达。

In [ ]:
plot_success_time_cdf(results_dict)
#横轴：成功时所用的步数，纵轴：累积分布概率（0→1）
#如果一条曲线更靠近左上角，说明该智能体能更快成功

图中应该只有 Prospect 一条曲线（因为其他两组没有成功数据）。
横轴是成功所需的步数，纵轴是累积比例。例如在40步时曲线达到0.5，表示50%的成功发生在40步以内。
你的 Prospect 平均成功时间66.79步，说明大部分成功需要约40-90步。

In [ ]:
plot_final_battery_heatmap(results_rational, ENV_CFG, "Rational")
plot_final_battery_heatmap(results_prospect, ENV_CFG, "Prospect")
plot_final_battery_heatmap(results_risk, ENV_CFG, "Risk-sensitive")
##将所有测试episode的最终位置作为小方格，计算该方格内episode的平均最终电量
#颜色越红表示到达该区域时电池剩余越多（通常表示更高效/更少耗电的路径）

In [ ]:
plot_trajectories(results_rational, ENV_CFG, "Rational")
plot_trajectories(results_prospect, ENV_CFG, "Prospect")
plot_trajectories(results_risk, ENV_CFG, "Risk-sensitive")

**为什么成功率低？**

我的实验中低成功率的主要原因是环境本身的高度复杂性。目标距离起点区域很远，起点和站点之间存在四个障碍物，且运动轨迹还会受到随机湍流的进一步干扰。此外，成功条件相当严格：无人机不仅必须到达站点区域，而且必须在第 140 步之前到达，并且电池电量必须大于 0.90。此外，大部分奖励是稀疏的：在大多数步骤中，智能体收到的几乎是均匀的负信号，只有在完全成功的情况下才会获得正奖励。因此，表格智能体很难学习到稳定且成功的策略。状态的离散化也会影响结果：控制精度会低于连续模型。因此，成功的轨迹很少见，这也是整体成功率仍然很低的原因。

Низкая успешность в моём эксперименте связана прежде всего с высокой сложностью самой среды. Цель находится далеко от стартовой области, между стартом и станцией расположены четыре препятствия, а движение дополнительно искажается случайным турбулентным ветром. Кроме того, условие успеха здесь довольно жёсткое: дрон должен не просто попасть в область станции, но и сделать это не позднее 140-го шага и с зарядом батареи выше 0.90. При этом основная часть награды является разреженной: в большинстве шагов агент получает почти одинаковый отрицательный сигнал, а положительная награда возникает только при полном успехе. Поэтому табличным агентам трудно выучить устойчивую успешную стратегию. Дополнительно на результат влияет и дискретизация состояния: управление становится менее точным, чем в непрерывной постановке. В результате успешные траектории оказываются редкими, и именно поэтому итоговая доля успеха остаётся низкой.

**为什么 Prospect 比 Rational 和 Risk-sensitive 更好？**

由于对负面结果的敏感性在本任务中尤为重要，因此前景Q学习表现最佳。在我们的环境中，无人机必须在一个包含障碍物、随机风向且成功条件严格的复杂空间中导航。标准的理性Q学习基于原始奖励进行学习，而前景智能体则使用基于前景理论的主观转换奖励。这意味着惩罚和局部决策失败的感知强度高于相应的积极线索。因此，该智能体能够更好地避免不良行为，并且有时仍能找到到达站点的路径。实验结果表明，前景Q学习是唯一在最终测试中达到非零成功率（8%）的智能体。此外，其平均能耗与理性智能体相当。因此，对于此环境，行为奖励转换比标准的理性准则和η=0.5的风险敏感熵方法更符合任务结构。

Prospect Q-learning показал лучший результат, потому что в этой задаче особенно важна чувствительность к отрицательным исходам. В нашей среде дрон должен пройти через сложное пространство с препятствиями, случайным ветром и жёстким условием успеха. Обычный рациональный Q-learning обучается на исходной награде, а Prospect-агент использует субъективно преобразованную награду по теории перспектив. Это означает, что штрафы и неудачные локальные решения воспринимаются сильнее, чем сопоставимые положительные сигналы. В результате агент лучше избегает плохих действий и иногда всё-таки находит маршрут к станции. По итогам эксперимента именно Prospect Q-learning оказался единственным агентом, который в финальном тестировании дал ненулевую долю успешных эпизодов — 8 %. При этом по среднему расходу энергии он не уступил рациональному агенту. Поэтому можно сказать, что для данной среды поведенческая трансформация награды лучше согласуется со структурой задачи, чем стандартный рациональный критерий и чем риск-чувствительный энтропийный подход с η = 0.5.

**为什么 risk-sensitive 反而没有表现更好？**

风险敏感型智能体表现不佳，是因为其在此问题中过于谨慎。熵准则中参数 η = 0.5 为正值时，智能体会表现出风险规避倾向：它会对结果不利且收益波动较大的轨迹施加更强烈的惩罚。理论上这可能是有益的，但在我们的实验环境中，成功本身就是一件罕见的事情：目标遥远，运动轨迹受随机风的影响，且发射点和目标站之间存在障碍物。在这些条件下，过于谨慎的策略并不能帮助无人机主动向目标移动。基于完整的实验结果，风险敏感型智能体没有取得任何成功，成功率为零，平均能耗在三个智能体中最高，平均最终电量也最低。因此，可以得出结论：在 η = 0.5 的情况下，此设置下的风险规避准则过于保守，其性能优于前景 Q 学习算法，后者在寻找罕见的成功轨迹方面表现更佳。

Риск-чувствительный агент не показал лучший результат, потому что в данной задаче его осторожность оказалась чрезмерной. Энтропийный критерий с положительным параметром η = 0.5 делает агента риск-аверсивным: он сильнее штрафует траектории с неблагоприятными исходами и с высокой вариативностью возврата. Теоретически это может быть полезно, но в нашей среде успех и так является редким событием: цель находится далеко, движение искажается случайным ветром, а между стартом и станцией расположены препятствия. В таких условиях слишком осторожная политика не помогает дрону активно продвигаться к цели. По итогам полного эксперимента риск-чувствительный агент не достиг ни одного успешного эпизода, показал нулевую успешность, самый высокий средний расход энергии среди трёх агентов и более низкий средний финальный заряд. Поэтому можно сказать, что при η = 0.5 риск-аверсивный критерий в этой постановке оказался слишком консервативным и уступил Prospect Q-learning, который лучше справился с поиском редких успешных траекторий.

**为什么要做 smoke-test？**

在正式发布之前，进行预测试（也称“冒烟测试”）是必要的。最初，我没有使用 15,000 个训练回合，而是使用了 2,000 个训练回合和 100 个测试回合进行训练，以便快速验证整个实验流程的正确性：包括环境创建、智能体训练、测试、指标计算以及保存结果以进行可视化。这一步骤使我能够提前验证代码运行正常，并且环境逻辑、Q 值更新或最终指标计算中不存在任何错误。此外，冒烟测试还让我初步了解了任务的复杂性：即使在这个阶段，也很明显成功的回合很少，但对前景和风险敏感的智能体有时能够找到成功的轨迹。因此，在冒烟测试之后，无需更改环境结构即可继续进行完整的实验。

Smoke-test был нужен как предварительный сокращённый эксперимент перед основным запуском. Сначала я запускала обучение не на 15 000 эпизодах, а на 2000 обучающих и 100 тестовых эпизодах, чтобы быстро проверить корректность всей экспериментальной цепочки: создание среды, обучение агента, тестирование, подсчёт метрик и сохранение результатов для визуализации. Такой шаг позволяет заранее убедиться, что код работает правильно и что нет ошибок в логике среды, обновлении Q-значений или вычислении итоговых показателей. Кроме того, smoke-test дал первое представление о сложности задачи: уже на этом этапе было видно, что успешные эпизоды редки, но Prospect- и risk-sensitive-агенты иногда всё же находят успешные траектории. Поэтому после smoke-test было разумно переходить к полному эксперименту без изменения структуры среды.

为什么要做 optimistic initialization？

为了鼓励智能体在学习初期更积极地探索状态-动作空间，需要采用乐观初始化。在我的实现中，所有智能体的初始Q值都设置为相同值，即Q0 = 5.0，这意味着智能体最初认为动作很有希望。这可以防止智能体过早地陷入随机获得的初始估计，并使其保持更长时间的探索倾向。这在我们的问题中尤为重要，因为环境复杂、奖励稀疏且成功回合很少。如果初始值是中性的或过低，智能体可能会过早地切换到较弱的局部策略，从而无法充分探索到达目标的可能路径。因此，这里使用乐观初始化作为一种​​额外的机制来刺激早期探索。所有三个智能体的初始值都相同，即Q0 = 5.0，并且对于风险敏感型智能体，U表使用指数变换进行统一初始化。

Оптимистическая инициализация (optimistic initialization) нужна для того, чтобы на раннем этапе обучения агент активнее исследовал пространство состояний и действий (state-action space). В моей реализации все начальные Q-значения задаются одинаково как Q0 = 5.0, то есть изначально агент считает действия достаточно перспективными. Благодаря этому он не слишком быстро “застревает” на случайно полученных первых оценках и дольше сохраняет склонность к исследованию. Это особенно важно в нашей задаче, потому что среда сложная, награда разреженная (sparse reward), а успешные эпизоды встречаются редко. Если бы начальные значения были нейтральными или слишком низкими, агент мог бы слишком рано перейти к слабой локальной стратегии и хуже изучить возможные маршруты к станции. Поэтому оптимистическая инициализация здесь использовалась как дополнительный механизм стимулирования раннего исследования. Для всех трёх агентов она была одинаковой, Q0 = 5.0, а для риск-чувствительного агента U-таблица инициализировалась согласованно через экспоненциальное преобразование.